# 01 - Data Check: Bank Marketing (bank-additional-full)

**Goal:** make sure the data is trustworthy before any analysis, and write down every decision.

**Business context:** phone campaign of a Portuguese bank selling a term deposit (May 2008 - Nov 2010). Target `y` = did the client subscribe.

**Outputs of this notebook**
- a decision table (issue found -> what I did -> why)
- a cleaned file `Data/bank_clean.csv` used later by SQL, Power BI and the model

Rules I follow: compute every number in code, keep original columns, explain *why* in comments.

In [1]:
from pathlib import Path
import pandas as pd

pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 140)

DATA_DIR = Path("../Data")
RAW_FILE = DATA_DIR / "bank-additional-full.csv"
CLEAN_FILE = DATA_DIR / "bank_clean.csv"

## 1. Load the file
The file uses `;` as separator, so `sep=";"` is required.

In [2]:
raw = pd.read_csv(RAW_FILE, sep=";")
print(f"rows: {raw.shape[0]:,} | columns: {raw.shape[1]}")
raw.head()

rows: 41,188 | columns: 21


,age,job,marital,education,default,housing,loan,contact,month,day_of_week,duration,campaign,pdays,previous,poutcome,emp.var.rate,cons.price.idx,cons.conf.idx,euribor3m,nr.employed,y
0,56,housemaid,married,basic.4y,no,no,no,telephone,may,mon,261,1,999,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,no
1,57,services,married,high.school,unknown,no,no,telephone,may,mon,149,1,999,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,no
2,37,services,married,high.school,no,yes,no,telephone,may,mon,226,1,999,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,no
3,40,admin.,married,basic.6y,no,no,no,telephone,may,mon,151,1,999,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,no
4,56,services,married,high.school,no,no,yes,telephone,may,mon,307,1,999,0,nonexistent,1.1,93.994,-36.4,4.857,5191.0,no


## 2. Column types, nulls, unique values
A quick look at what each column holds. Real nulls are checked here; the dataset hides missing values as the text `unknown` (section 5).

In [3]:
overview = pd.DataFrame({
    "dtype": raw.dtypes.astype(str),
    "nulls": raw.isna().sum(),
    "unique_values": raw.nunique(),
})
overview

,dtype,nulls,unique_values
age,int64,0,78
job,object,0,12
marital,object,0,4
education,object,0,8
default,object,0,3
housing,object,0,3
loan,object,0,3
contact,object,0,2
month,object,0,10
day_of_week,object,0,5


## 3. Target balance
If few clients say yes, accuracy becomes a misleading score. I compute the baseline now so it can be quoted later.

In [4]:
yes_rate_raw = (raw["y"] == "yes").mean()
baseline_accuracy = 1 - yes_rate_raw

print(raw["y"].value_counts().to_string())
print(f"\nyes rate: {yes_rate_raw:.1%}")
print(f"baseline accuracy (always predict 'no'): {baseline_accuracy:.1%}")

y
no     36548
yes     4640

yes rate: 11.3%
baseline accuracy (always predict 'no'): 88.7%


**Why this matters:** any model has to beat the always-"no" baseline, and accuracy alone cannot show that. Later I use recall, precision, AUC and lift.

## 4. Duplicate rows
There is no customer ID, so identical rows might be two real calls to similar clients. I check how many there are and whether removing them changes anything.

In [5]:
dup_mask = raw.duplicated(keep="first")
print(f"rows identical to an earlier row (all {raw.shape[1]} columns): {dup_mask.sum()}")
print(f"share of file: {dup_mask.mean():.3%}")

yes_rate_dedup = (raw.loc[~dup_mask, "y"] == "yes").mean()
print(f"yes rate before: {yes_rate_raw:.3%} | after dropping duplicates: {yes_rate_dedup:.3%}")

raw[raw.duplicated(keep=False)].sort_values(list(raw.columns)).head(8)

rows identical to an earlier row (all 21 columns): 12
share of file: 0.029%
yes rate before: 11.265% | after dropping duplicates: 11.266%


,age,job,marital,education,default,housing,loan,contact,month,day_of_week,duration,campaign,pdays,previous,poutcome,emp.var.rate,cons.price.idx,cons.conf.idx,euribor3m,nr.employed,y
28476,24,services,single,high.school,no,yes,no,cellular,apr,tue,114,1,999,0,nonexistent,-1.8,93.075,-47.1,1.423,5099.1,no
28477,24,services,single,high.school,no,yes,no,cellular,apr,tue,114,1,999,0,nonexistent,-1.8,93.075,-47.1,1.423,5099.1,no
14155,27,technician,single,professional.course,no,no,no,cellular,jul,mon,331,2,999,0,nonexistent,1.4,93.918,-42.7,4.962,5228.1,no
14234,27,technician,single,professional.course,no,no,no,cellular,jul,mon,331,2,999,0,nonexistent,1.4,93.918,-42.7,4.962,5228.1,no
18464,32,technician,single,professional.course,no,yes,no,cellular,jul,thu,128,1,999,0,nonexistent,1.4,93.918,-42.7,4.968,5228.1,no
18465,32,technician,single,professional.course,no,yes,no,cellular,jul,thu,128,1,999,0,nonexistent,1.4,93.918,-42.7,4.968,5228.1,no
32505,35,admin.,married,university.degree,no,yes,no,cellular,may,fri,348,4,999,0,nonexistent,-1.8,92.893,-46.2,1.313,5099.1,no
32516,35,admin.,married,university.degree,no,yes,no,cellular,may,fri,348,4,999,0,nonexistent,-1.8,92.893,-46.2,1.313,5099.1,no


**Decision:** drop them. They are a tiny share of the file, the yes rate barely moves, and several pairs sit right next to each other in the file, which fits accidental duplication. I cannot prove that without a customer ID, so this stays a documented judgment, not a certainty.

## 5. Hidden missing values: `unknown`

In [6]:
cat_cols = raw.select_dtypes(include="object").columns.drop("y")
unknown_rows = (raw[cat_cols] == "unknown").sum()

unknown_table = pd.DataFrame({
    "unknown_rows": unknown_rows,
    "pct_of_rows": (unknown_rows / len(raw) * 100).round(1),
})
unknown_table = unknown_table[unknown_table["unknown_rows"] > 0].sort_values("unknown_rows", ascending=False)
unknown_table

,unknown_rows,pct_of_rows
default,8597,20.9
education,1731,4.2
housing,990,2.4
loan,990,2.4
job,330,0.8
marital,80,0.2


In [7]:
# Does "unknown" behave differently from known answers? If yes, it carries information and should not be thrown away.
rows = []
for col in unknown_table.index:
    is_unknown = raw[col] == "unknown"
    rows.append({
        "column": col,
        "yes_rate_if_unknown_%": round((raw.loc[is_unknown, "y"] == "yes").mean() * 100, 1),
        "yes_rate_if_known_%": round((raw.loc[~is_unknown, "y"] == "yes").mean() * 100, 1),
    })
pd.DataFrame(rows).set_index("column")

,yes_rate_if_unknown_%,yes_rate_if_known_%
column,,
default,5.2,12.9
education,14.5,11.1
housing,10.8,11.3
loan,10.8,11.3
job,11.2,11.3
marital,15.0,11.3


In [8]:
# housing and loan have the same number of unknowns: are they the same rows?
same_rows = ((raw["housing"] == "unknown") == (raw["loan"] == "unknown")).all()
print("housing and loan are unknown on exactly the same rows:", same_rows)

housing and loan are unknown on exactly the same rows: True


**Decision:** keep `unknown` as its own category instead of deleting rows or guessing values.
- `default` is the clearest case: its unknown group has a very different yes rate from the known group, so it carries signal.
- `housing` and `loan` are missing together, which suggests one cause (the answer was not recorded in the call), not random noise.

## 6. Range and consistency checks

In [9]:
range_checks = pd.Series({
    "age min": raw["age"].min(),
    "age max": raw["age"].max(),
    "duration == 0 (rows)": int((raw["duration"] == 0).sum()),
    "campaign max (contacts in this campaign)": raw["campaign"].max(),
    "previous max": raw["previous"].max(),
    "pdays == 999 (% of rows)": round((raw["pdays"] == 999).mean() * 100, 1),
}, name="value")
range_checks.to_frame()

,value
age min,17.0
age max,98.0
duration == 0 (rows),4.0
campaign max (contacts in this campaign),56.0
previous max,7.0
pdays == 999 (% of rows),96.3


In [10]:
# pdays = 999 means "not previously contacted". But `previous` and `poutcome` also describe prior contact. Do they agree?
print("pdays == 999 vs previous == 0")
print(pd.crosstab(raw["pdays"] == 999, raw["previous"] == 0, rownames=["pdays==999"], colnames=["previous==0"]))
print()
print("poutcome vs previous == 0")
print(pd.crosstab(raw["poutcome"], raw["previous"] == 0, colnames=["previous==0"]))

pdays == 999 vs previous == 0
previous==0  False  True 
pdays==999               
False         1515      0
True          4110  35563

poutcome vs previous == 0
previous==0  False  True 
poutcome                 
failure       4252      0
nonexistent      0  35563
success       1373      0


**What I found:** `previous` and `poutcome` agree perfectly (no prior contact = `nonexistent`). `pdays` does not: some clients were contacted before (`previous > 0`) but `pdays` still shows 999.

**Decision:** define `previously_contacted` from `previous > 0` (consistent with `poutcome`), and do not use `pdays == 999` as the definition. `pdays` stays in the file as it came.

`duration == 0` rows are real calls that never connected, so they stay. `duration` itself is kept for descriptive reporting only; it is **not allowed in any predictive model** because it is known only after the call ends.

## 7. Row order and campaign year
The documentation says rows are ordered by date (May 2008 - Nov 2010) but there is no year column. A month name alone mixes up up to three different years, so I check the order and rebuild the year.

In [11]:
month_blocks = raw["month"].ne(raw["month"].shift()).cumsum()
blocks = raw.groupby(month_blocks)["month"].agg(first_month="first", rows="size")
blocks.T

month,1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19,20,21,22,23,24,25,26
first_month,may,jun,jul,aug,oct,nov,dec,mar,apr,may,jun,jul,aug,sep,oct,nov,dec,mar,apr,may,jun,jul,aug,sep,oct,nov
rows,7763,4374,6685,5175,67,3616,10,282,2458,5794,715,178,770,267,447,357,172,264,174,212,229,311,233,303,204,128


In [12]:
month_num = raw["month"].map({m: i for i, m in enumerate(
    ["jan","feb","mar","apr","may","jun","jul","aug","sep","oct","nov","dec"], start=1)})

# The year goes up by one each time the month number falls (dec -> mar means a new year started).
year_offset = (month_num < month_num.shift(fill_value=month_num.iloc[0])).cumsum()
campaign_year = 2008 + year_offset

print(campaign_year.value_counts().sort_index().to_string())
print("\nfirst row:", campaign_year.iloc[0], raw["month"].iloc[0], "| last row:", campaign_year.iloc[-1], raw["month"].iloc[-1])

month
2008    27690
2009    11440
2010     2058

first row: 2008 may | last row: 2010 nov


**Decision:** add `year` (inferred from file order) and `campaign_period` (YYYY-MM). The order is chronological, so a later time-based train/test split can simply use row position. The year is an inference from file order, not an original field, and I label it that way.

## 8. Build the clean table
Original columns are kept untouched. New columns are added with readable names.

In [13]:
clean = raw.copy()
clean["year"] = campaign_year
clean["campaign_period"] = clean["year"].astype(str) + "-" + month_num.astype(str).str.zfill(2)

# drop exact duplicates AFTER year is built, so row order stays intact
clean = clean.drop_duplicates(keep="first").copy()
clean.insert(0, "source_row", clean.index + 1)   # position in the original file = time order

clean["y_flag"] = (clean["y"] == "yes").astype(int)
clean["previously_contacted"] = (clean["previous"] > 0).astype(int)

clean["age_band"] = pd.cut(clean["age"], bins=[0, 24, 29, 39, 49, 59, 120],
                           labels=["<25", "25-29", "30-39", "40-49", "50-59", "60+"])
clean["campaign_band"] = pd.cut(clean["campaign"], bins=[0, 1, 2, 3, 4, 5, 7, 10, 100],
                                labels=["1", "2", "3", "4", "5", "6-7", "8-10", "11+"])

print(f"rows: {len(clean):,} (removed {len(raw) - len(clean)})")
clean[["source_row", "age", "age_band", "campaign", "campaign_band", "previous", "previously_contacted", "year", "campaign_period", "y", "y_flag"]].head()

rows: 41,176 (removed 12)


,source_row,age,age_band,campaign,campaign_band,previous,previously_contacted,year,campaign_period,y,y_flag
0,1,56,50-59,1,1,0,0,2008,2008-05,no,0
1,2,57,50-59,1,1,0,0,2008,2008-05,no,0
2,3,37,30-39,1,1,0,0,2008,2008-05,no,0
3,4,40,40-49,1,1,0,0,2008,2008-05,no,0
4,5,56,50-59,1,1,0,0,2008,2008-05,no,0


## 9. Decision table

In [14]:
decisions = pd.DataFrame([
    ["Duplicate rows", f"{dup_mask.sum()} rows identical to an earlier row", "Dropped", "Tiny share, yes rate unchanged, most likely entry duplicates"],
    ["Hidden missing values", f"{int(unknown_table['unknown_rows'].sum()):,} 'unknown' cells across {len(unknown_table)} columns", "Kept as its own category", "Unknown differs from known (clearest in default); deleting would lose signal"],
    ["pdays = 999", f"{(raw['pdays'] == 999).mean():.1%} of rows", "pdays untouched; previously_contacted built from previous > 0", "pdays disagrees with previous/poutcome; those two agree"],
    ["duration", "Only known after the call", "Kept for description, excluded from models", "Data leakage"],
    ["Target imbalance", f"yes = {yes_rate_dedup:.1%}", "Judge models by recall, precision, AUC, lift", f"Always-'no' already scores {1 - yes_rate_dedup:.1%} accuracy"],
    ["No year column", f"{clean['year'].nunique()} years hidden behind month names", "Added year and campaign_period (inferred from order)", "Month alone mixes years"],
    ["Continuous variables", "age and campaign have many distinct values", "Added age_band and campaign_band; kept originals", "Readable groups for SQL and Power BI"],
], columns=["Issue", "What I found", "What I did", "Why"])
decisions

,Issue,What I found,What I did,Why
0,Duplicate rows,12 rows identical to an earlier row,Dropped,"Tiny share, yes rate unchanged, most likely en..."
1,Hidden missing values,"12,718 'unknown' cells across 6 columns",Kept as its own category,Unknown differs from known (clearest in defaul...
2,pdays = 999,96.3% of rows,pdays untouched; previously_contacted built fr...,pdays disagrees with previous/poutcome; those ...
3,duration,Only known after the call,"Kept for description, excluded from models",Data leakage
4,Target imbalance,yes = 11.3%,"Judge models by recall, precision, AUC, lift",Always-'no' already scores 88.7% accuracy
5,No year column,3 years hidden behind month names,Added year and campaign_period (inferred from ...,Month alone mixes years
6,Continuous variables,age and campaign have many distinct values,Added age_band and campaign_band; kept originals,Readable groups for SQL and Power BI


## 10. Save the cleaned file and re-load it as a check

In [15]:
clean.to_csv(CLEAN_FILE, index=False)

check = pd.read_csv(CLEAN_FILE)
assert len(check) == len(clean), "row count changed after saving"
assert check["y_flag"].sum() == (clean["y"] == "yes").sum(), "subscriber count changed after saving"

print(f"saved: {CLEAN_FILE.name} | rows: {len(check):,} | columns: {check.shape[1]}")
print(f"overall yes rate: {check['y_flag'].mean():.1%}")
print(f"clients previously contacted: {check['previously_contacted'].mean():.1%}")

saved: bank_clean.csv | rows: 41,176 | columns: 28
overall yes rate: 11.3%
clients previously contacted: 13.7%


## Summary
- The file is clean enough to use: no real nulls, duplicates removed, `unknown` kept on purpose.
- Two things to remember in every later notebook: `duration` is off-limits for prediction, and accuracy must be compared with the always-"no" baseline.
- Next: `02_eda.ipynb` - yes rate by group (always with group size), starting with campaign fatigue.